In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes as required
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# ---- metric.csv: group by cmdb_id and kpi_name and compute global stats ----
# Ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Group and aggregate
grouped = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_stats = grouped.agg(
    count='count',
    min='min',
    p50=lambda s: s.median(),
    p95=lambda s: s.quantile(0.95),
    max='max'
).reset_index()

# Reorder columns as requested
metric_stats = metric_stats[['cmdb_id', 'kpi_name', 'count', 'p50', 'p95', 'min', 'max']]

# Top 20 groups sorted by count
metric_top20 = metric_stats.sort_values('count', ascending=False).head(20)

# ---- trace.csv: unique trace_name values with counts, top 20 by count ----
trace_counts = trace_df.groupby('trace_name').size().reset_index(name='count')
trace_top20 = trace_counts.sort_values('count', ascending=False).head(20)

# ---- log.csv: unique log_name values with counts, top 20 by count ----
log_counts = log_df.groupby('log_name').size().reset_index(name='count')
log_top20 = log_counts.sort_values('count', ascending=False).head(20)

# ---- error_logs.csv: total row count, top5 cmdb_id counts, earliest and latest rows ----
error_total_count = len(error_df)

# Top 5 distinct cmdb_id values with counts
error_top5_cmdb = error_df['cmdb_id'].value_counts().head(5).reset_index()
error_top5_cmdb.columns = ['cmdb_id', 'count']

# Earliest and latest timestamp rows (selected compact columns)
# Keep timestamp as UTC datetime (already parsed), show only timestamp, cmdb_id, message
if error_total_count > 0:
    error_earliest_row = error_df.nsmallest(1, 'timestamp')[['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)
    error_latest_row = error_df.nlargest(1, 'timestamp')[['timestamp', 'cmdb_id', 'message']].reset_index(drop=True)
else:
    error_earliest_row = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])
    error_latest_row = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])

# Final compact outputs
metric_top20, trace_top20, log_top20, error_total_count, error_top5_cmdb, error_earliest_row, error_latest_row

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files:\n\n"
    "Metrics (metric.csv):\n"
    "- Global stats were computed per (cmdb_id, kpi_name). The top groups by sample count all have 25 samples.\n"
    "- Representative top groups (count=25):\n"
    "  * carts — cpu: p50≈3.108, p95≈7.557, min≈1.698, max≈15.740\n"
    "  * carts — mem: p50≈2.0855e8, p95≈2.1042e8, min≈1.9436e8, max≈2.1058e8\n"
    "  * carts-db — diskio: p50≈1.767e6, p95≈1.933e6, min≈1.517e6, max≈1.979e6\n"
    "  * catalogue — cpu: p50≈0.1756, p95≈0.1842, min≈0.1547, max≈0.1908\n"
    "  (The full top-20 groups table was produced; all shown groups had count=25.)\n\n"
    "Traces (trace.csv):\n"
    "- No trace_name values were found (empty result).\n\n"
    "Logs (log.csv):\n"
    "- Two log_name values present, each with 228 samples:\n"
    "  * log.error_count (count=228)\n"
    "  * log.row_count   (count=228)\n\n"
    "Error logs (error_logs.csv):\n"
    "- Total rows: 0 (no error_log entries).\n"
    "- Therefore no top cmdb_id counts and no earliest/latest error rows to report.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count           p50           p95           min           max
0          carts         cpu     25  3.108134e+00  7.556864e+00  1.698345e+00  1.574011e+01
1          carts  latency-50     25  1.282368e-02  1.589716e-02  9.534802e-03  1.897828e-02
2          carts  latency-90     25  2.334535e-02  3.008061e-02  2.181554e-02  5.032418e-02
3          carts         mem     25  2.085548e+08  2.104214e+08  1.943634e+08  2.105784e+08
4          carts      socket     25  1.055000e+01  1.197000e+01  9.183333e+00  1.273333e+01
5          carts    workload     25  8.142083e+00  8.593180e+00  5.977667e+00  8.686767e+00
6       carts-db         cpu     25  2.084763e+00  3.035954e+00  1.546074e+00  3.103291e+00
7       carts-db      diskio     25  1.767348e+06  1.933417e+06  1.517086e+06  1.978872e+06
8       carts-db         mem     25  7.099453e+07  7.385667e+07  6.959923e+07  7.425594e+07
9       carts-db      socket     25  6.000000e+00  6.866667e+00  6.000000e+00  7.000000e+00
10     catalogue         cpu     25  1.755877e-01  1.842038e-01  1.546866e-01  1.907844e-01
11     catalogue  latency-50     25  3.044874e-03  3.094803e-03  3.010999e-03  3.103385e-03
12     catalogue  latency-90     25  4.680773e-03  4.770646e-03  4.619799e-03  4.786093e-03
13     catalogue         mem     25  5.800960e+06  5.900807e+06  4.673536e+06  5.905613e+06
14     catalogue      socket     25  6.000000e+00  7.000000e+00  4.000000e+00  7.000000e+00
15     catalogue    workload     25  4.101050e+00  4.289497e+00  3.901067e+00  4.442350e+00
16  catalogue-db         cpu     25  2.017428e-01  2.100491e-01  1.862499e-01  2.255356e-01
17  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
18  catalogue-db         mem     25  4.304912e+08  4.304943e+08  4.302356e+08  4.304955e+08
19  catalogue-db      socket     25  4.000000e+00  4.000000e+00  3.000000e+00  4.000000e+00, Empty DataFrame
Columns: [trace_name, count]
Index: [],           log_name  count
0  log.error_count    228
1    log.row_count    228, 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

# Reuse existing DataFrames: metric_df, log_df, error_df (from previous run)
import pandas as pd
from pandas import Timestamp

# Ensure timestamp columns are parsed and value columns numeric
for df, valcol in ((metric_df, 'value'), (log_df, 'value')):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df[valcol] = pd.to_numeric(df[valcol], errors='coerce')

if 'timestamp' in error_df.columns:
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Parameters
allowed_cmdb = ['carts','catalogue','orders','payment','user']
incident_start = pd.to_datetime('2024-01-20 07:34:54', utc=True)
incident_end   = pd.to_datetime('2024-01-20 08:04:54', utc=True)

# --------------------
# METRICS: global P95 per (cmdb_id, kpi_name) for allowed cmdbs
# --------------------
df_allowed = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()

# Identify allowed cmdbs with zero samples (skipped)
allowed_present_cmdbs = df_allowed['cmdb_id'].unique().tolist()
skipped_cmdbs = sorted(list(set(allowed_cmdb) - set(allowed_present_cmdbs)))
metric_skipped = pd.DataFrame({'skipped_cmdb': skipped_cmdbs}) if skipped_cmdbs else pd.DataFrame(columns=['skipped_cmdb'])

# If no allowed data at all, prepare empty results
if df_allowed.shape[0] == 0:
    metric_result = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window'])
else:
    # compute global p95 per group (global across entire file, before any time filtering)
    global_p95_df = df_allowed.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

    # Filter rows to incident window (only allowed cmdbs already in df_allowed)
    metric_window = df_allowed[(df_allowed['timestamp'] >= incident_start) & (df_allowed['timestamp'] <= incident_end)].copy()

    # Merge global_p95 onto window rows so each row has its threshold
    if metric_window.shape[0] > 0:
        metric_win_merged = metric_window.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')
        metric_win_merged['is_anomaly'] = metric_win_merged['value'] >= metric_win_merged['global_p95']

        # aggregated stats per group from window
        agg_anom = metric_win_merged.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
            anomaly_count_in_window = ('is_anomaly', 'sum'),
            earliest_anomaly_time_UTC = ('timestamp', lambda s: s[metric_win_merged.loc[s.index, 'is_anomaly']].min() if s.shape[0]>0 else pd.NaT),
            max_value_in_window = ('value', 'max')
        )
        # The lambda above uses index alignment to get min timestamp only among anomaly rows.
        # However pandas may warn; compute earliest_anomaly separately for clarity:
        anom_rows = metric_win_merged[metric_win_merged['is_anomaly']]
        if anom_rows.shape[0] > 0:
            earliest_anom = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_time_UTC'})
        else:
            earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_time_UTC'])

        max_vals = metric_win_merged.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].max().rename(columns={'value':'max_value_in_window'})
        counts = metric_win_merged.groupby(['cmdb_id','kpi_name'], as_index=False)['is_anomaly'].sum().rename(columns={'is_anomaly':'anomaly_count_in_window'})

        window_stats = counts.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left').merge(max_vals, on=['cmdb_id','kpi_name'], how='left')

    else:
        # no rows in window
        window_stats = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window'])

    # Left join global_p95_df with window_stats to include groups that had global data but zero window rows
    metric_result = global_p95_df.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
    metric_result['anomaly_count_in_window'] = metric_result['anomaly_count_in_window'].fillna(0).astype(int)
    metric_result['max_value_in_window'] = metric_result['max_value_in_window']  # may be NaN if no window rows
    metric_result['earliest_anomaly_time_UTC'] = pd.to_datetime(metric_result['earliest_anomaly_time_UTC'], utc=True)

    # Keep only requested columns and sort by anomaly_count desc, limit 20
    metric_result = metric_result[['cmdb_id','kpi_name','global_p95','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window']].sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# --------------------
# LOGS: global P95 per (cmdb_id, log_name) and anomaly counts in window
# --------------------
if 'log_name' not in log_df.columns or log_df.shape[0] == 0:
    log_result = pd.DataFrame(columns=['cmdb_id','log_name','global_p95','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window'])
    log_skipped = pd.DataFrame(columns=['note'])
else:
    # compute global p95 per (cmdb_id, log_name) across entire log file
    global_p95_log = log_df.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

    # identify any (cmdb_id,log_name) combos expected? We only consider those present in global_p95_log.
    # No additional "requested groups" specified, so skipped groups is empty.
    log_skipped = pd.DataFrame(columns=['skipped'])  # none

    # filter log rows to incident window
    log_window = log_df[(log_df['timestamp'] >= incident_start) & (log_df['timestamp'] <= incident_end)].copy()

    if log_window.shape[0] > 0:
        # merge global p95 to window rows
        log_win_merged = log_window.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')
        log_win_merged['is_anomaly'] = log_win_merged['value'] >= log_win_merged['global_p95']

        anom_rows_log = log_win_merged[log_win_merged['is_anomaly']]
        if anom_rows_log.shape[0] > 0:
            earliest_anom_log = anom_rows_log.groupby(['cmdb_id','log_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_time_UTC'})
        else:
            earliest_anom_log = pd.DataFrame(columns=['cmdb_id','log_name','earliest_anomaly_time_UTC'])

        counts_log = log_win_merged.groupby(['cmdb_id','log_name'], as_index=False)['is_anomaly'].sum().rename(columns={'is_anomaly':'anomaly_count_in_window'})
        max_vals_log = log_win_merged.groupby(['cmdb_id','log_name'], as_index=False)['value'].max().rename(columns={'value':'max_value_in_window'})

        window_stats_log = counts_log.merge(earliest_anom_log, on=['cmdb_id','log_name'], how='left').merge(max_vals_log, on=['cmdb_id','log_name'], how='left')
    else:
        window_stats_log = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window'])

    # Combine with global thresholds to ensure groups with no window rows are included with zero counts
    log_result = global_p95_log.merge(window_stats_log, on=['cmdb_id','log_name'], how='left')
    log_result['anomaly_count_in_window'] = log_result['anomaly_count_in_window'].fillna(0).astype(int)
    log_result['earliest_anomaly_time_UTC'] = pd.to_datetime(log_result['earliest_anomaly_time_UTC'], utc=True)
    log_result = log_result[['cmdb_id','log_name','global_p95','anomaly_count_in_window','earliest_anomaly_time_UTC','max_value_in_window']].sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# --------------------
# ERROR LOGS: total count and rows inside incident window (up to 10)
# --------------------
error_count = len(error_df)
error_in_window = error_df[(error_df['timestamp'] >= incident_start) & (error_df['timestamp'] <= incident_end)].sort_values('timestamp')[['timestamp','cmdb_id','message']].head(10).reset_index(drop=True)

# Final compact outputs
metric_result, metric_skipped, log_result, log_skipped, error_count, error_in_window

```
Out[2]:
```
```python
summary = (
    "Summary (UTC):\n\n"
    "Metric analysis (restricted to carts, catalogue, orders, payment, user):\n"
    "- No allowed cmdb_id groups were skipped (all requested groups that have samples were processed).\n"
    "- Top metric anomalies in the incident window [2024-01-20 07:34:54, 2024-01-20 08:04:54]:\n"
    "  1) orders / diskio — global P95=0.0, anomaly_count=24, earliest=2024-01-20 07:38:00+00:00, max_in_window=0.0\n"
    "  2) catalogue / socket — global P95=7.0, anomaly_count=10, earliest=2024-01-20 07:52:00+00:00, max_in_window=7.0\n"
    "  3) payment / socket — global P95=6.0, anomaly_count=4, earliest=2024-01-20 07:58:00+00:00, max_in_window=6.0\n"
    "  4) Several groups had smaller counts (mostly anomaly_count=2) including carts (mem, socket, cpu, latency-50, latency-90, workload),\n"
    "     catalogue (cpu, mem, latency-50, latency-90, workload), and orders (cpu, mem, latency-50, latency-90, socket). \n"
    "- Interpretation: orders.diskio shows the largest number of points exceeding its global P95 in the window (24 points). Times reported are UTC.\n\n"
    "Log analysis:\n"
    "- Global P95 computed per (cmdb_id, log_name). Top log anomalies in the window (by anomaly count):\n"
    "  * Many services show log.error_count anomalies with anomaly_count=25 (e.g., catalogue, user, payment, queue-master, orders, front-end, shipping).\n"
    "  * carts-db log.error_count: anomaly_count=24 (earliest=2024-01-20 07:38:00+00:00).\n"
    "  * carts log.row_count: global P95=100.0, anomaly_count=4, earliest=2024-01-20 07:54:00+00:00, max_in_window=101.\n"
    "  * Several other services show elevated log.row_count (example: queue-master, session-db, front-end, orders, payment).\n\n"
    "Error logs:\n"
    "- Total error_logs rows in file: 0. No error_log rows exist inside the incident window.\n\n"
    "Overall concise finding:\n"
    "- The largest metric-level anomaly count in the window is orders / diskio (24 points exceeding its global P95).\n"
    "- Multiple services show repeated log.error_count anomalies during the same window (many with count=25), and carts-db / carts show notable log anomalies too.\n"
    "- No error_log entries are present.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_time_UTC  max_value_in_window
0      orders      diskio  0.000000e+00                       24 2024-01-20 07:38:00+00:00         0.000000e+00
1   catalogue      socket  7.000000e+00                       10 2024-01-20 07:52:00+00:00         7.000000e+00
2     payment      socket  6.000000e+00                        4 2024-01-20 07:58:00+00:00         6.000000e+00
3       carts         mem  2.104214e+08                        2 2024-01-20 08:00:00+00:00         2.105784e+08
4       carts      socket  1.197000e+01                        2 2024-01-20 07:37:00+00:00         1.273333e+01
5       carts  latency-50  1.589716e-02                        2 2024-01-20 07:37:00+00:00         1.897828e-02
6       carts         cpu  7.556864e+00                        2 2024-01-20 07:37:00+00:00         1.574011e+01
7   catalogue         cpu  1.842038e-01                        2 2024-01-20 07:39:00+00:00         1.907844e-01
8       carts    workload  8.593180e+00                        2 2024-01-20 07:47:00+00:00         8.686767e+00
9   catalogue  latency-90  4.770646e-03                        2 2024-01-20 07:39:00+00:00         4.786093e-03
10  catalogue  latency-50  3.094803e-03                        2 2024-01-20 07:39:00+00:00         3.103385e-03
11  catalogue         mem  5.900807e+06                        2 2024-01-20 08:00:00+00:00         5.905613e+06
12  catalogue    workload  4.289497e+00                        2 2024-01-20 07:37:00+00:00         4.442350e+00
13     orders         cpu  7.700270e+01                        2 2024-01-20 07:51:00+00:00         7.712773e+01
14      carts  latency-90  3.008061e-02                        2 2024-01-20 07:37:00+00:00         5.032418e-02
15     orders  latency-50  8.412805e-02                        2 2024-01-20 07:52:00+00:00         1.323988e-01
16     orders  latency-90  5.402414e-01                        2 2024-01-20 07:51:00+00:00         5.613962e-01
17     orders      socket  2.491667e+01                        2 2024-01-20 07:52:00+00:00         2.498182e+01
18     orders         mem  9.159246e+08                        2 2024-01-20 07:58:00+00:00         9.869709e+08
19    payment         cpu  9.756158e-02                        2 2024-01-20 07:47:00+00:00         9.802719e-02, Empty DataFrame
Columns: [skipped_cmdb]
Index: [],          cmdb_id         log_name  global_p95  anomaly_count_in_window earliest_anomaly_time_UTC  max_value_in_window
0      catalogue  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
1           user  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
2        payment  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
3   queue-master  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
4         orders  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
5      front-end  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
6       shipping  log.error_count         0.0                       25 2024-01-20 07:37:00+00:00                    0
7       carts-db  log.error_count         0.0                       24 2024-01-20 07:38:00+00:00                    0
8       carts-db    log.row_count         1.0                       24 2024-01-20 07:38:00+00:00                    2
9          carts  log.error_count         0.0                       21 2024-01-20 07:38:00+00:00                    0
10    session-db  log.error_count         0.0                        5 2024-01-20 07:38:00+00:00                    0
11    session-db    log.row_count         5.0                        5 2024-01-20 07:38:00+00:00                    5
12         carts    log.row_count       100.0                        4 2024-01-20 07:54:00+00:00                  101
13      shipping    log.row_count        67.0                        3 2024-01-20 07:41:00+00:00                   68
14       user-db  log.error_count         0.0                        3 2024-01-20 07:46:00+00:00                    0
15  queue-master    log.row_count       603.0                        3 2024-01-20 07:41:00+00:00                  612
16        orders    log.row_count       134.0                        3 2024-01-20 07:41:00+00:00                  136
17       payment    log.row_count       107.0                        3 2024-01-20 07:41:00+00:00                  108
18     catalogue    log.row_count       170.6                        2 2024-01-20 07:47:00+00:00                  171
19     front-end    log.row_count      2076.6                        2 2024-01-20 07:47:00+00:00                 2187, Empty DataFrame
Columns: [skipped]
Index: [], 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[3]:

# Reuse existing DataFrames from prior kernel state: metric_df, log_df
import pandas as pd

# Parameters
allowed_cmdb = ['carts','catalogue','orders','payment','user']
incident_start = pd.to_datetime('2024-01-20 07:34:54', utc=True)
incident_end   = pd.to_datetime('2024-01-20 08:04:54', utc=True)

# Ensure timestamps and numeric values are correct
for df, timecol, valcol in ((metric_df, 'timestamp', 'value'), (log_df, 'timestamp', 'value')):
    if timecol in df.columns:
        df[timecol] = pd.to_datetime(df[timecol], unit='s', utc=True)
    df[valcol] = pd.to_numeric(df[valcol], errors='coerce')

# Helper to compute longest consecutive run information for a series of timestamps (UTC)
def longest_consecutive_run(ts_series):
    # ts_series: pd.Series of Timestamps sorted ascending, no duplicates expected
    if ts_series.empty:
        return 0, pd.NaT
    ts = ts_series.reset_index(drop=True)
    diffs = ts.diff()
    # start of a run where diff != 1 minute or first element
    new_run = (diffs != pd.Timedelta(minutes=1))
    run_id = new_run.cumsum()
    runs = ts.groupby(run_id).agg(start_time=('first'), length=('count'))
    # select the run with max length; if multiple, choose earliest start_time
    runs = runs.reset_index(drop=True)
    max_len = int(runs['length'].max())
    earliest_start = runs[runs['length'] == max_len]['start_time'].min()
    return max_len, pd.to_datetime(earliest_start, utc=True)

# ------------------------
# METRIC ANALYSIS
# ------------------------
# Restrict to allowed cmdbs
df_m = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()

# Compute global P95 per (cmdb_id, kpi_name) across full series
if df_m.shape[0] == 0:
    metric_result = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','total_anomaly_count','longest_run','earliest_start_of_longest_run_UTC','max_value_in_window'])
    metric_skipped = pd.DataFrame({'skipped_group': []})
else:
    global_p95_m = df_m.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

    # Filter window rows
    df_m_win = df_m[(df_m['timestamp'] >= incident_start) & (df_m['timestamp'] <= incident_end)].copy()

    # Identify groups that have no samples in window and will be skipped
    all_groups = set(zip(global_p95_m['cmdb_id'], global_p95_m['kpi_name']))
    win_groups = set(zip(df_m_win['cmdb_id'], df_m_win['kpi_name']))
    skipped_groups = sorted(list(all_groups - win_groups))
    metric_skipped = pd.DataFrame({'skipped_group': ['%s|%s' % g for g in skipped_groups[:20]]})

    if df_m_win.shape[0] == 0:
        metric_result = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','total_anomaly_count','longest_run','earliest_start_of_longest_run_UTC','max_value_in_window'])
    else:
        # merge global threshold into window rows
        df_m_win = df_m_win.merge(global_p95_m, on=['cmdb_id','kpi_name'], how='left')

        # anomaly rule:
        # - if global_p95 > 0: anomaly if value > global_p95 (strict)
        # - if global_p95 == 0: anomaly if value > 0
        df_m_win['is_anom'] = ((df_m_win['global_p95'] > 0) & (df_m_win['value'] > df_m_win['global_p95'])) | \
                              ((df_m_win['global_p95'] == 0) & (df_m_win['value'] > 0))

        # group and compute stats
        records = []
        grouped = df_m_win.groupby(['cmdb_id','kpi_name'], sort=False)
        for (cmdb, kpi), g in grouped:
            g = g.sort_values('timestamp')
            total_anom_count = int(g['is_anom'].sum())
            # longest consecutive run among anomaly timestamps
            anom_ts = g.loc[g['is_anom'], 'timestamp']
            longest_run, earliest_start = longest_consecutive_run(anom_ts)
            max_value = g['value'].max()
            # Only include groups that have at least one sample in window (we already filtered)
            records.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'global_p95': g['global_p95'].iloc[0],
                'total_anomaly_count': total_anom_count,
                'longest_run': int(longest_run),
                'earliest_start_of_longest_run_UTC': earliest_start,
                'max_value_in_window': max_value
            })

        metric_result = pd.DataFrame.from_records(records)
        # Skip groups with zero samples? All here had window samples by construction.
        # Keep only groups where at least one anomaly occurred? The user asked to skip groups with no samples in window,
        # not those with zero anomalies. So keep all, then sort and limit.
        metric_result = metric_result.sort_values(['longest_run','total_anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# ------------------------
# LOG ANALYSIS (restricted to allowed cmdbs)
# ------------------------
df_l = log_df[log_df['cmdb_id'].isin(allowed_cmdb)].copy()

if df_l.shape[0] == 0:
    log_result = pd.DataFrame(columns=['cmdb_id','log_name','global_p95','total_anomaly_count','longest_run','earliest_start_of_longest_run_UTC','max_value_in_window'])
    log_skipped = pd.DataFrame({'skipped_group': []})
else:
    global_p95_l = df_l.groupby(['cmdb_id','log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_p95'})

    df_l_win = df_l[(df_l['timestamp'] >= incident_start) & (df_l['timestamp'] <= incident_end)].copy()

    all_log_groups = set(zip(global_p95_l['cmdb_id'], global_p95_l['log_name']))
    win_log_groups = set(zip(df_l_win['cmdb_id'], df_l_win['log_name']))
    skipped_log_groups = sorted(list(all_log_groups - win_log_groups))
    log_skipped = pd.DataFrame({'skipped_group': ['%s|%s' % g for g in skipped_log_groups[:20]]})

    if df_l_win.shape[0] == 0:
        log_result = pd.DataFrame(columns=['cmdb_id','log_name','global_p95','total_anomaly_count','longest_run','earliest_start_of_longest_run_UTC','max_value_in_window'])
    else:
        df_l_win = df_l_win.merge(global_p95_l, on=['cmdb_id','log_name'], how='left')
        df_l_win['is_anom'] = ((df_l_win['global_p95'] > 0) & (df_l_win['value'] > df_l_win['global_p95'])) | \
                              ((df_l_win['global_p95'] == 0) & (df_l_win['value'] > 0))

        records_l = []
        grouped_l = df_l_win.groupby(['cmdb_id','log_name'], sort=False)
        for (cmdb, lname), g in grouped_l:
            g = g.sort_values('timestamp')
            total_anom_count = int(g['is_anom'].sum())
            anom_ts = g.loc[g['is_anom'], 'timestamp']
            longest_run, earliest_start = longest_consecutive_run(anom_ts)
            max_value = g['value'].max()
            records_l.append({
                'cmdb_id': cmdb,
                'log_name': lname,
                'global_p95': g['global_p95'].iloc[0],
                'total_anomaly_count': total_anom_count,
                'longest_run': int(longest_run),
                'earliest_start_of_longest_run_UTC': earliest_start,
                'max_value_in_window': max_value
            })

        log_result = pd.DataFrame.from_records(records_l)
        log_result = log_result.sort_values(['longest_run','total_anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Final compact outputs
metric_result, metric_skipped, log_result, log_skipped

```
Out[3]:
```
Summary (UTC):

Metrics (restricted to carts, catalogue, orders, payment, user; incident window 2024-01-20 07:34:54–08:04:54):
- No requested groups were skipped (all groups present were evaluated).
- Top groups by longest consecutive anomaly run (rule: if global_p95>0 then value>global_p95, else if global_p95==0 then value>0). All times shown are UTC.
  1. carts / cpu — global_p95=7.556864, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:37:00, max_value_in_window=15.74011
  2. carts / mem — global_p95=2.104214e8, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 08:00:00, max_value=2.105784e8
  3. catalogue / mem — global_p95=5.900807e6, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 08:00:00, max_value=5.905613e6
  4. orders / latency-50 — global_p95=0.08412805, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:52:00, max_value=0.1323988
  5. orders / latency-90 — global_p95=0.5402414, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:51:00, max_value=0.5613962
  6. payment / mem — global_p95=5.260493e6, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:44:00, max_value=5.343915e6
  7. payment / workload — global_p95=2.133110, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:46:00, max_value=2.188783
  8. user / cpu — global_p95=1.114205, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 07:46:00, max_value=1.137096
  9. user / mem — global_p95=7.845547e6, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 08:00:00, max_value=7.918387e6
 10. user / socket — global_p95=22.12, total_anomaly_count=2, longest_run=2 min, earliest_start=2024-01-20 08:00:00, max_value=23.0
- Many other metric groups showed short anomaly runs (mostly 1–2 minutes) and low counts (mostly 2 anomalies). No group showed a long continuous anomaly run longer than 2 minutes.

Logs (restricted to same cmdbs; same incident window):
- No requested log groups were skipped.
- Top log groups by longest consecutive anomaly run:
  1. catalogue / log.row_count — global_p95=170.6, total_anomaly_count=2, longest_run=1 min, earliest_start=2024-01-20 07:47:00, max_value=171
  2. user / log.row_count — global_p95=627.8, total_anomaly_count=2, longest_run=1 min, earliest_start=2024-01-20 07:41:00, max_value=656
  3. orders / log.row_count — global_p95=134.0, total_anomaly_count=1, longest_run=1 min, earliest_start=2024-01-20 07:47:00, max_value=136
  4. payment / log.row_count — global_p95=107.0, total_anomaly_count=1, longest_run=1 min, earliest_start=2024-01-20 07:47:00, max_value=108
  5. carts / log.row_count — global_p95=100.0, total_anomaly_count=1, longest_run=1 min, earliest_start=2024-01-20 07:59:00, max_value=101
- For log.error_count across these services, global_p95=0.0 and no anomalies were observed in the window (values remained 0).
- Overall log anomalies are short (mostly single-minute events) and sparse compared with metrics.

Key takeaway:
- Metric anomalies during the incident window are short-lived (mostly 1–2 minutes) across several services (carts, catalogue, orders, payment, user). No prolonged (>2 min) continuous anomaly runs were found.
- Log.row_count shows some brief spikes; log.error_count did not show anomalies for the evaluated services in the window.
- All reported times are UTC and the results are limited to the specified services and incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name    global_p95  total_anomaly_count  longest_run earliest_start_of_longest_run_UTC  max_value_in_window
0       carts         cpu  7.556864e+00                    2            2         2024-01-20 07:37:00+00:00         1.574011e+01
1       carts         mem  2.104214e+08                    2            2         2024-01-20 08:00:00+00:00         2.105784e+08
2   catalogue         mem  5.900807e+06                    2            2         2024-01-20 08:00:00+00:00         5.905613e+06
3      orders  latency-50  8.412805e-02                    2            2         2024-01-20 07:52:00+00:00         1.323988e-01
4      orders  latency-90  5.402414e-01                    2            2         2024-01-20 07:51:00+00:00         5.613962e-01
5     payment         mem  5.260493e+06                    2            2         2024-01-20 07:44:00+00:00         5.343915e+06
6     payment    workload  2.133110e+00                    2            2         2024-01-20 07:46:00+00:00         2.188783e+00
7        user         cpu  1.114205e+00                    2            2         2024-01-20 07:46:00+00:00         1.137096e+00
8        user         mem  7.845547e+06                    2            2         2024-01-20 08:00:00+00:00         7.918387e+06
9        user      socket  2.212000e+01                    2            2         2024-01-20 08:00:00+00:00         2.300000e+01
10      carts  latency-50  1.589716e-02                    2            1         2024-01-20 07:37:00+00:00         1.897828e-02
11      carts  latency-90  3.008061e-02                    2            1         2024-01-20 07:37:00+00:00         5.032418e-02
12      carts      socket  1.197000e+01                    2            1         2024-01-20 07:37:00+00:00         1.273333e+01
13      carts    workload  8.593180e+00                    2            1         2024-01-20 07:47:00+00:00         8.686767e+00
14  catalogue         cpu  1.842038e-01                    2            1         2024-01-20 07:39:00+00:00         1.907844e-01
15  catalogue  latency-50  3.094803e-03                    2            1         2024-01-20 07:39:00+00:00         3.103385e-03
16  catalogue  latency-90  4.770646e-03                    2            1         2024-01-20 07:39:00+00:00         4.786093e-03
17  catalogue    workload  4.289497e+00                    2            1         2024-01-20 07:37:00+00:00         4.442350e+00
18     orders         cpu  7.700270e+01                    2            1         2024-01-20 07:51:00+00:00         7.712773e+01
19     orders         mem  9.159246e+08                    2            1         2024-01-20 07:58:00+00:00         9.869709e+08, Empty DataFrame
Columns: [skipped_group]
Index: [],      cmdb_id         log_name  global_p95  total_anomaly_count  longest_run earliest_start_of_longest_run_UTC  max_value_in_window
0  catalogue    log.row_count       170.6                    2            1         2024-01-20 07:47:00+00:00                  171
1       user    log.row_count       627.8                    2            1         2024-01-20 07:41:00+00:00                  656
2     orders    log.row_count       134.0                    1            1         2024-01-20 07:47:00+00:00                  136
3    payment    log.row_count       107.0                    1            1         2024-01-20 07:47:00+00:00                  108
4      carts    log.row_count       100.0                    1            1         2024-01-20 07:59:00+00:00                  101
5  catalogue  log.error_count         0.0                    0            0                               NaT                    0
6     orders  log.error_count         0.0                    0            0                               NaT                    0
7    payment  log.error_count         0.0                    0            0                               NaT                    0
8       user  log.error_count         0.0                    0            0                               NaT                    0
9      carts  log.error_count         0.0                    0            0                               NaT                    0, Empty DataFrame
Columns: [skipped_group]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing metric_df and log_df; ensure timestamps and numeric values are correct
incident_start = pd.to_datetime('2024-01-20 07:34:54', utc=True)
incident_end   = pd.to_datetime('2024-01-20 08:04:54', utc=True)

for df in (metric_df, log_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Metrics: carts / cpu within incident window
mask_cpu_all = (metric_df['cmdb_id'] == 'carts') & (metric_df['kpi_name'] == 'cpu')
cpu_series_all = metric_df.loc[mask_cpu_all, 'value']

if cpu_series_all.empty:
    cpu_global_p95 = float('nan')
else:
    cpu_global_p95 = float(cpu_series_all.quantile(0.95))

# Windowed rows for carts cpu
mask_cpu_win = mask_cpu_all & (metric_df['timestamp'] >= incident_start) & (metric_df['timestamp'] <= incident_end)
cpu_rows = metric_df.loc[mask_cpu_win, ['timestamp','value']].sort_values('timestamp').head(20).reset_index(drop=True)

# Summary for the group
cpu_total_points = int(cpu_rows.shape[0])
cpu_earliest = cpu_rows['timestamp'].min() if cpu_total_points>0 else pd.NaT
cpu_latest   = cpu_rows['timestamp'].max() if cpu_total_points>0 else pd.NaT
cpu_max_value = cpu_rows['value'].max() if cpu_total_points>0 else float('nan')

# Compute consecutive minutes where value > global_p95 (strict greater)
if cpu_total_points == 0 or pd.isna(cpu_global_p95):
    cpu_longest_run = 0
    cpu_longest_start = pd.NaT
else:
    anom_ts = cpu_rows.loc[cpu_rows['value'] > cpu_global_p95, 'timestamp'].sort_values().reset_index(drop=True)
    if anom_ts.empty:
        cpu_longest_run = 0
        cpu_longest_start = pd.NaT
    else:
        diffs = anom_ts.diff()
        new_run = (diffs != pd.Timedelta(minutes=1))
        run_id = new_run.cumsum()
        runs = anom_ts.groupby(run_id).agg(start_time=('first'), length=('count')).reset_index(drop=True)
        cpu_longest_run = int(runs['length'].max())
        cpu_longest_start = pd.to_datetime(runs[runs['length'] == cpu_longest_run]['start_time'].min(), utc=True)

cpu_summary = pd.DataFrame([{
    'global_p95': cpu_global_p95,
    'total_points_in_window': cpu_total_points,
    'earliest_in_window_UTC': cpu_earliest,
    'latest_in_window_UTC': cpu_latest,
    'max_value_in_window': cpu_max_value,
    'longest_run_minutes': cpu_longest_run,
    'start_time_longest_run_UTC': cpu_longest_start
}])

# 2) Logs: carts within incident window
mask_logs_carts_win = (log_df['cmdb_id'] == 'carts') & (log_df['timestamp'] >= incident_start) & (log_df['timestamp'] <= incident_end)
carts_log_rows = log_df.loc[mask_logs_carts_win, ['timestamp','log_name','value']].sort_values('timestamp').head(20).reset_index(drop=True)

# Counts per log_name inside the window
if carts_log_rows.shape[0] == 0:
    log_counts = pd.DataFrame(columns=['log_name','count'])
else:
    log_counts = carts_log_rows.groupby('log_name', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).reset_index(drop=True)

# How many log.error_count rows have value > 0
mask_error_count_positive = mask_logs_carts_win & (log_df['log_name'] == 'log.error_count') & (log_df['value'] > 0)
log_error_count_positive = int(log_df.loc[mask_error_count_positive].shape[0])

# Final compact outputs
cpu_rows, cpu_summary, carts_log_rows, log_counts, log_error_count_positive

```
Out[4]:
```
Summary (UTC):

1) Metrics — carts / cpu (incident window 2024-01-20 07:34:54–08:04:54)
- Returned rows (up to 20) show per-minute cpu values from 2024-01-20 07:37:00 through 07:56:00 (20 points).
- Global P95 (computed from full series): 7.556864
- In-window summary:
  - Total points in window: 20
  - Earliest timestamp in-window: 2024-01-20 07:37:00+00:00
  - Latest timestamp in-window: 2024-01-20 07:56:00+00:00
  - Max value in-window: 15.740108
  - Longest consecutive anomaly run (value > global_p95): 2 minutes
    - Start of longest run: 2024-01-20 07:37:00+00:00

2) Logs — carts (same incident window)
- Returned rows (up to 20) show interleaved log.error_count and log.row_count samples between 07:38 and 07:48 UTC.
- Counts in-window by log_name:
  - log.error_count: 10 rows
  - log.row_count: 10 rows
- Number of log.error_count rows with value > 0: 0 (all error_count values are 0 in the window)

All timestamps and summaries above are in UTC.

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp      value
0  2024-01-20 07:37:00+00:00  15.740108
1  2024-01-20 07:38:00+00:00   7.594617
2  2024-01-20 07:39:00+00:00   5.305390
3  2024-01-20 07:40:00+00:00   7.405851
4  2024-01-20 07:41:00+00:00   4.817799
5  2024-01-20 07:42:00+00:00   4.133795
6  2024-01-20 07:43:00+00:00   3.756531
7  2024-01-20 07:44:00+00:00   5.292739
8  2024-01-20 07:45:00+00:00   3.773719
9  2024-01-20 07:46:00+00:00   2.838149
10 2024-01-20 07:47:00+00:00   3.108134
11 2024-01-20 07:48:00+00:00   3.470129
12 2024-01-20 07:49:00+00:00   5.798697
13 2024-01-20 07:50:00+00:00   3.237217
14 2024-01-20 07:51:00+00:00   1.974887
15 2024-01-20 07:52:00+00:00   2.025910
16 2024-01-20 07:53:00+00:00   1.862524
17 2024-01-20 07:54:00+00:00   2.022266
18 2024-01-20 07:55:00+00:00   2.675500
19 2024-01-20 07:56:00+00:00   1.799787,    global_p95  total_points_in_window    earliest_in_window_UTC      latest_in_window_UTC  max_value_in_window  longest_run_minutes start_time_longest_run_UTC
0    7.556864                      20 2024-01-20 07:37:00+00:00 2024-01-20 07:56:00+00:00            15.740108                    2  2024-01-20 07:37:00+00:00,                    timestamp         log_name  value
0  2024-01-20 07:38:00+00:00  log.error_count      0
1  2024-01-20 07:38:00+00:00    log.row_count     25
2  2024-01-20 07:39:00+00:00  log.error_count      0
3  2024-01-20 07:39:00+00:00    log.row_count     50
4  2024-01-20 07:40:00+00:00  log.error_count      0
5  2024-01-20 07:40:00+00:00    log.row_count     25
6  2024-01-20 07:42:00+00:00  log.error_count      0
7  2024-01-20 07:42:00+00:00    log.row_count     25
8  2024-01-20 07:43:00+00:00  log.error_count      0
9  2024-01-20 07:43:00+00:00    log.row_count     75
10 2024-01-20 07:44:00+00:00    log.row_count     75
11 2024-01-20 07:44:00+00:00  log.error_count      0
12 2024-01-20 07:45:00+00:00  log.error_count      0
13 2024-01-20 07:45:00+00:00    log.row_count     50
14 2024-01-20 07:46:00+00:00  log.error_count      0
15 2024-01-20 07:46:00+00:00    log.row_count     25
16 2024-01-20 07:47:00+00:00  log.error_count      0
17 2024-01-20 07:47:00+00:00    log.row_count     50
18 2024-01-20 07:48:00+00:00  log.error_count      0
19 2024-01-20 07:48:00+00:00    log.row_count     25,           log_name  count
0  log.error_count     10
1    log.row_count     10, 0)```
```